# DML Semi-real (Job Corps) — Unified Notebook (Shared Dataset)

This notebook consolidates the **KNN / LASSO / NN** semi-real DML runs into one place.

How it works:
- Select `ALGO = "KNN" | "LASSO" | "NN"` at the top to run exactly one method.
- For each seed, we generate and shuffle the dataset once and cache it.
  The same seed always uses the same `(X, T, Y)` across algorithms, so
  method comparisons are not confounded by data-generation randomness.

Note on reproducibility:
- Neural network training can be nondeterministic depending on framework and hardware.
- The data itself is fixed per seed, and we set as many RNG seeds as possible.


In [1]:
# ============================================================
# 0) Config (edit here only)
# ============================================================
ALGO = "LASSO"     # "KNN" | "LASSO" | "NN"
K_RUNS = 100    # Number of Monte Carlo runs
BASE_SEED = 1   # First seed (subsequent runs use BASE_SEED + k)

# Paths:
# - Colab: mount Drive and point BASE_DIR to your repo location.
# - Local: use the current working directory.
USE_COLAB_DRIVE = True

import os, sys, pathlib, random, logging
import numpy as np
import pandas as pd

try:
    if USE_COLAB_DRIVE:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        BASE_DIR = pathlib.Path("/content/drive/MyDrive/Colab Notebooks/CTE_Codes/DML_methods")
    else:
        raise RuntimeError("skip colab")
except Exception:
    BASE_DIR = pathlib.Path(".").resolve()

# Supplement import path
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
SUPP_DIR = BASE_DIR / "Supplement"
if SUPP_DIR.exists() and str(SUPP_DIR) not in sys.path:
    sys.path.insert(0, str(SUPP_DIR))

import Supplement  # noqa: E402

# Output directory (CSV estimates + MISE summaries)
RESULTS_DIR = BASE_DIR / "Data_and_Results" / "Estimates"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR: {BASE_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)sZ | %(levelname)s | %(filename)s:%(lineno)d | %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S",
)
logger = logging.getLogger(__name__)

# Default treatment grid (matches original scripts)
T_GRID_DEFAULT = np.arange(160, 2001, 40)


Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1305: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:434.)
  _C._set_default_tensor_type(t)


BASE_DIR: /content/drive/MyDrive/Colab Notebooks/CTE_Codes/DML_methods
RESULTS_DIR: /content/drive/MyDrive/Colab Notebooks/CTE_Codes/DML_methods/Data_and_Results/Estimates


In [2]:
# ============================================================
# 1) Shared helpers: RNG control & semi-synthetic outcomes
# ============================================================

def set_all_seeds(seed: int) -> None:
    """Set as many RNG seeds as possible for reproducibility.

    Note: the dataset itself is cached per seed; this mainly stabilizes
    model training (especially for NN runs).
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

    # If torch is available, fix its RNGs and determinism flags.
    try:
        import torch  # type: ignore
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass
    except Exception:
        pass


def gen_semi_y(mu_hat: np.ndarray, g: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Generate semi-synthetic outcomes: Y = mu_hat + e * g, e in {-1, +1}."""
    n = len(mu_hat)
    e = rng.choice([-1.0, 1.0], size=n)
    return mu_hat + e * g


def mise_against(est_beta, h_star_vals) -> float:
    """Compute MISE between an estimated curve and ground truth on the grid."""
    return float(np.mean((np.asarray(est_beta) - np.asarray(h_star_vals)) ** 2))


def summarize_list(x_list):
    """Return (mean, std, standard error) for a list of scalars."""
    arr = np.asarray(x_list, dtype=float)
    mean = float(arr.mean()) if len(arr) > 0 else 0.0
    std = float(arr.std(ddof=1)) if len(arr) > 1 else 0.0
    se = float(std / np.sqrt(len(arr))) if len(arr) > 0 else 0.0
    return mean, std, se


In [3]:
# ============================================================
# 2) Data loading (one-time)
# ============================================================

def load_jobcorps_data(base_dir: pathlib.Path):
    """Load Job Corps data and precomputed GRF components.

    Returns:
        X (pd.DataFrame): Covariates after one-hot encoding.
        T (pd.Series): Treatment variable (column "d").
        Y_emp (pd.Series): Observed outcome (column "y"; not used for semi-synthetic Y).
        mu_hat (np.ndarray): Baseline prediction from GRF.
        g (np.ndarray): Semi-synthetic noise scale from GRF.
        t_grid (np.ndarray): Grid for evaluating h*(t).
        h_star_vals (np.ndarray): Ground-truth h*(t) on the grid.
    """
    emp_dir = base_dir / "Data_and_Results"
    data_path = emp_dir / "emp_app.csv"
    semi_path = emp_dir / "semi-syn data grf.csv"
    h_star_path = emp_dir / "h_star_grf_empapp.csv"

    logger.info(f"Loading emp_app.csv from: {data_path}")
    data = pd.read_csv(data_path, index_col=0)

    # Fixed shuffle to match the original script.
    data = data.sample(frac=1, random_state=20)

    # One-hot encoding for categorical (int64) columns.
    data = pd.concat(
        [
            data.select_dtypes(exclude=["int64"]),
            pd.get_dummies(
                data.select_dtypes("int64").astype("category"),
                drop_first=True,
                dtype=float,
            ),
        ],
        axis=1,
    )

    X = data.drop(["d", "y"], axis=1)
    T = data["d"]
    Y_emp = data["y"]

    logger.info(f"Loading semi-synthetic components from: {semi_path}")
    semi_df = pd.read_csv(semi_path, index_col=0)
    if not np.array_equal(semi_df.index.values, data.index.values):
        semi_df = semi_df.loc[data.index]
    mu_hat = semi_df["mu_hat_grf"].to_numpy()
    g = semi_df["g_grf"].to_numpy()

    logger.info(f"Loading h_star ground truth from: {h_star_path}")
    h_star_df = pd.read_csv(h_star_path)

    # Prefer t-grid from file; otherwise fall back to the default grid.
    if "t" in h_star_df.columns:
        t_grid = h_star_df["t"].to_numpy()
    else:
        t_grid = T_GRID_DEFAULT

    h_star_vals = h_star_df["h_star"].to_numpy()

    # Sanity check
    if len(T) != len(mu_hat) or len(T) != len(g):
        logger.warning(
            f"Length mismatch: len(T)={len(T)}, len(mu_hat)={len(mu_hat)}, len(g)={len(g)}. "
            "Check that the source files are aligned and sorted consistently."
        )

    return X, T, Y_emp, mu_hat, g, t_grid, h_star_vals


X, T, Y_emp, mu_hat, g, t_list, h_star_vals = load_jobcorps_data(BASE_DIR)
n_obs = len(T)

print("Loaded shapes:")
print("  X:", X.shape)
print("  T:", T.shape)
print("  mu_hat:", mu_hat.shape, "g:", g.shape)
print("  t_list:", len(t_list), "h_star:", len(h_star_vals))


Loaded shapes:
  X: (4024, 138)
  T: (4024,)
  mu_hat: (4024,) g: (4024,)
  t_list: 47 h_star: 47


In [4]:
# ============================================================
# 3) Shared dataset cache (one (X, T, Y) per seed)
# ============================================================

def make_seed_cache(
    X_df: pd.DataFrame,
    T_s: pd.Series,
    mu_hat: np.ndarray,
    g: np.ndarray,
    seeds: list,
):
    """Create a per-seed cache of shuffled datasets.

    Each seed produces the same semi-synthetic Y and the same permutation,
    so all algorithms see identical data for that seed.
    """
    cache = {}
    n = len(T_s)
    for seed in seeds:
        rng_sim = np.random.default_rng(int(seed))

        # Semi-synthetic outcome (shared across algorithms)
        Y_syn = gen_semi_y(mu_hat, g, rng_sim)

        # Shared shuffle
        perm = rng_sim.permutation(n)

        X_k_df = X_df.iloc[perm].reset_index(drop=True)
        T_k_s = T_s.iloc[perm].reset_index(drop=True)
        Y_k = np.asarray(Y_syn[perm], float)

        cache[int(seed)] = {
            "perm": perm,
            "X_df": X_k_df,
            "T_s": T_k_s,
            "Y": Y_k,
            "X_np": np.asarray(X_k_df, float),
            "T_np": np.asarray(T_k_s, float),
        }
    return cache


seed_list = [BASE_SEED + k for k in range(K_RUNS)]
SEED_CACHE = make_seed_cache(X, T, mu_hat, g, seed_list)

print(f"Prepared shared dataset cache for {len(SEED_CACHE)} seeds.")


Prepared shared dataset cache for 100 seeds.


In [5]:
# ============================================================
# 4A) KNN version (original logic, shared data cache)
# ============================================================

def run_knn_simulation(seed_cache, t_list, h_star_vals):
    """Run the KNN/NN-based DML pipeline using cached data."""
    # Original hyperparameters and bandwidth rules.
    h_rule = np.std(T) * 3 * (n_obs ** (-0.2))
    h_first = 2 * h_rule
    L, u = 5, 0.5

    models = [
        Supplement.NeuralNet1k_emp_app(k=138, lr=0.15, momentum=0.9, epochs=100, weight_decay=0.05),
        Supplement.NeuralNet2_emp_app(k=138, lr=0.05, momentum=0.3, epochs=100, weight_decay=0.15),
    ]
    basis = False

    stage2_mise = []
    stage2_beta = []
    seeds = []

    logger.info(f"Starting {len(seed_cache)} simulations (KNN)...")

    for i, seed in enumerate(sorted(seed_cache.keys())):
        seeds.append(seed)
        print("", flush=True)
        print(f"[Progress] Processing Simulation {i+1}/{len(seed_cache)} (Seed: {seed})", flush=True)

        # Fix training RNGs (data are already cached per seed)
        set_all_seeds(seed)

        X_k = seed_cache[seed]["X_df"]
        T_k = seed_cache[seed]["T_s"]
        Y_k = seed_cache[seed]["Y"]

        DDML_Wrapper = Supplement.NN_DDMLCT  # KNN uses the NN-based DDML wrapper

        # Stage 1 (two bandwidths)
        m1 = DDML_Wrapper(models[0], models[1])
        m1.fit(X_k, T_k, Y_k, t_list, L, h=h_first, basis=basis, standardize=True)

        m2 = DDML_Wrapper(models[0], models[1])
        m2.fit(X_k, T_k, Y_k, t_list, L, h=h_first * u, basis=basis, standardize=True)

        Bt = (m1.beta - m2.beta) / ((m1.h ** 2) * (1 - (u ** 2)))
        h_star_ml = np.mean(((m2.Vt / (4 * (Bt ** 2))) ** 0.2) * (m1.n ** -0.2))

        # Stage 2 (final bandwidth)
        h_second = 0.8 * h_star_ml
        m_final = DDML_Wrapper(models[0], models[1])
        m_final.fit(X_k, T_k, Y_k, t_list, L, h=h_second, basis=basis, standardize=True)

        mise2 = mise_against(m_final.beta, h_star_vals)
        stage2_mise.append(mise2)
        stage2_beta.append(np.asarray(m_final.beta, dtype=float))

        if (i + 1) % 10 == 0:
            logger.info(f"Simulation {i+1}/{len(seed_cache)} completed.")

    mean2, std2, se2 = summarize_list(stage2_mise)
    print("\n" + "=" * 50)
    print(f"RESULTS (KNN, K={len(seed_cache)}) - Second Stage Only")
    print("=" * 50)
    print(f"Mean MISE : {mean2:.6f}")
    print(f"Std MISE  : {std2:.6f}")
    print(f"SE MISE   : {se2:.6f}")
    print("=" * 50 + "\n")

    return {
        "t_grid": t_list,
        "beta_mat": np.vstack(stage2_beta),
        "mise_list": stage2_mise,
        "seeds": seeds,
    }


In [6]:
# ============================================================
# 4B) LASSO version (original logic, shared data cache)
# ============================================================

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.linear_model import Lasso

class LassoWithL2Normalize(BaseEstimator, RegressorMixin):
    """LASSO wrapper matching the original setup.

    - Centers X and y
    - Scales features by L2 norm
    - Fits LASSO without an intercept
    """
    def __init__(self, alpha=1.0, max_iter=10000, tol=1e-4, random_state=None):
        self.alpha = alpha
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self._model = None

    def fit(self, X, y):
        X = np.asarray(X, float)
        y = np.asarray(y, float)

        self.X_mean_ = X.mean(axis=0)
        self.y_mean_ = y.mean()
        X_centered = X - self.X_mean_
        y_centered = y - self.y_mean_

        self.X_scale_ = np.linalg.norm(X_centered, axis=0)
        self.X_scale_[self.X_scale_ == 0] = 1.0
        X_norm = X_centered / self.X_scale_

        self._model = Lasso(
            alpha=self.alpha,
            max_iter=self.max_iter,
            tol=self.tol,
            fit_intercept=False,
            random_state=self.random_state,
        )
        self._model.fit(X_norm, y_centered)
        return self

    def predict(self, X):
        X = np.asarray(X, float)
        X_norm = (X - self.X_mean_) / self.X_scale_
        return self._model.predict(X_norm) + self.y_mean_


def run_lasso_simulation(seed_cache, t_list, h_star_vals):
    """Run the LASSO-based DML pipeline using cached data."""
    h_rule = np.std(T) * 3 * (len(T) ** (-0.2))
    h_first = 2 * h_rule
    L, u = 5, 0.5

    model_lasso1 = LassoWithL2Normalize(alpha=0.00069944, max_iter=10000, tol=0.0001)
    model_lasso2 = LassoWithL2Normalize(alpha=0.000160472, max_iter=10000, tol=0.0001)
    models = [model_lasso1, model_lasso2]

    stage2_mise = []
    stage2_beta = []
    seeds = []

    logger.info(f"Starting {len(seed_cache)} simulations (LASSO)...")

    for i, seed in enumerate(sorted(seed_cache.keys())):
        seeds.append(seed)
        print("", flush=True)
        print(f"[Progress] Processing Simulation {i+1}/{len(seed_cache)} (Seed: {seed})", flush=True)

        set_all_seeds(seed)

        X_k = seed_cache[seed]["X_np"]
        T_k = seed_cache[seed]["T_np"]
        Y_k = seed_cache[seed]["Y"]

        model1 = Supplement.DDMLCT(models[0], models[1])
        model1.fit(X_k, T_k, Y_k, t_list, L, h=h_first, basis=True, standardize=True)

        model2 = Supplement.DDMLCT(models[0], models[1])
        model2.fit(X_k, T_k, Y_k, t_list, L, h=h_first * u, basis=True, standardize=True)

        Bt = (model1.beta - model2.beta) / ((model1.h**2) * (1 - (u**2)))
        h_star_ml = np.mean(((model2.Vt / (4 * (Bt**2))) ** 0.2) * (model1.n**-0.2))

        h_second = 0.8 * h_star_ml
        model_final = Supplement.DDMLCT(models[0], models[1])
        model_final.fit(X_k, T_k, Y_k, t_list, L, h=h_second, basis=True, standardize=True)

        mise2 = mise_against(model_final.beta, h_star_vals)
        stage2_mise.append(mise2)
        stage2_beta.append(np.asarray(model_final.beta, dtype=float))

        if (i + 1) % 10 == 0:
            logger.info(f"Simulation {i+1}/{len(seed_cache)} completed.")

    mean2, std2, se2 = summarize_list(stage2_mise)
    print("\n" + "=" * 50)
    print(f"RESULTS (LASSO, K={len(seed_cache)}) - Second Stage Only")
    print("=" * 50)
    print(f"Mean MISE : {mean2:.6f}")
    print(f"Std MISE  : {std2:.6f}")
    print(f"SE MISE   : {se2:.6f}")
    print("=" * 50 + "\n")

    return {
        "t_grid": t_list,
        "beta_mat": np.vstack(stage2_beta),
        "mise_list": stage2_mise,
        "seeds": seeds,
    }


In [7]:
# ============================================================
# 4C) NN version (original logic, shared data cache)
# ============================================================

def run_nn_simulation(seed_cache, t_list, h_star_vals):
    """Run the NN-based DML pipeline using cached data."""
    n_obs_local = len(T)

    h_rule = np.std(T) * 3 * (n_obs_local ** (-0.2))
    h_first = 2 * h_rule
    L, u = 5, 0.5

    model_nn1 = Supplement.NeuralNet1_emp_app(k=139, lr=0.15, momentum=0.9, epochs=100, weight_decay=0.05)
    model_nn2 = Supplement.NeuralNet2_emp_app(k=138, lr=0.05, momentum=0.3, epochs=100, weight_decay=0.15)

    stage2_mise = []
    stage2_beta = []
    seeds = []

    logger.info(f"Starting {len(seed_cache)} simulations (NN)...")

    for i, seed in enumerate(sorted(seed_cache.keys())):
        seeds.append(seed)
        print("", flush=True)
        print(f"[Progress] Processing Simulation {i+1}/{len(seed_cache)} (Seed: {seed})", flush=True)

        set_all_seeds(seed)

        X_k = seed_cache[seed]["X_df"]
        T_k = seed_cache[seed]["T_s"]
        Y_k = seed_cache[seed]["Y"]

        DDML_Class = Supplement.DDMLCT

        m1 = DDML_Class(model_nn1, model_nn2)
        m1.fit(X_k, T_k, Y_k, t_list, L, h=h_first, basis=False, standardize=True)

        m2 = DDML_Class(model_nn1, model_nn2)
        m2.fit(X_k, T_k, Y_k, t_list, L, h=h_first * u, basis=False, standardize=True)

        Bt = (m1.beta - m2.beta) / ((m1.h**2) * (1 - (u**2)))
        h_star_ml = np.mean(((m2.Vt / (4 * (Bt**2))) ** 0.2) * (m1.n**-0.2))

        h_second = 0.8 * h_star_ml
        m_final = DDML_Class(model_nn1, model_nn2)
        m_final.fit(X_k, T_k, Y_k, t_list, L, h=h_second, basis=False, standardize=True)

        mise2 = mise_against(m_final.beta, h_star_vals)
        stage2_mise.append(mise2)
        stage2_beta.append(np.asarray(m_final.beta, dtype=float))

        if (i + 1) % 10 == 0:
            logger.info(f"Simulation {i+1}/{len(seed_cache)} completed.")

    mean2, std2, se2 = summarize_list(stage2_mise)
    print("\n" + "=" * 50)
    print(f"RESULTS (NN, K={len(seed_cache)}) - Second Stage Only")
    print("=" * 50)
    print(f"Mean MISE : {mean2:.6f}")
    print(f"Std MISE  : {std2:.6f}")
    print(f"SE MISE   : {se2:.6f}")
    print("=" * 50 + "\n")

    return {
        "t_grid": t_list,
        "beta_mat": np.vstack(stage2_beta),
        "mise_list": stage2_mise,
        "seeds": seeds,
    }


In [8]:
# ============================================================
# 5) Dispatch + Save (run only the selected ALGO)
# ============================================================

ALGO_UP = ALGO.strip().upper()
if ALGO_UP not in {"KNN", "LASSO", "NN"}:
    raise ValueError(f"ALGO must be one of KNN/LASSO/NN, got: {ALGO}")

if ALGO_UP == "KNN":
    results = run_knn_simulation(SEED_CACHE, t_list, h_star_vals)
elif ALGO_UP == "LASSO":
    results = run_lasso_simulation(SEED_CACHE, t_list, h_star_vals)
else:
    results = run_nn_simulation(SEED_CACHE, t_list, h_star_vals)

first_seed = results["seeds"][0]
last_seed = results["seeds"][-1]

# ---- Save beta estimates (all seeds + mean/se) ----
t_cols = [f"t_{int(t)}" for t in results["t_grid"]]
df_beta = pd.DataFrame(results["beta_mat"], columns=t_cols)
df_beta.insert(0, "seed", results["seeds"])

mean_row = pd.DataFrame([results["beta_mat"].mean(axis=0)], columns=t_cols)
mean_row.insert(0, "seed", "mean")

se_row = pd.DataFrame(
    [results["beta_mat"].std(axis=0, ddof=1) / np.sqrt(len(results["seeds"]))],
    columns=t_cols,
)
se_row.insert(0, "seed", "se")

df_beta_final = pd.concat([df_beta, mean_row, se_row], ignore_index=True)

out_beta = RESULTS_DIR / f"estimates_{ALGO_UP}_seed{first_seed}_to_seed{last_seed}.csv"
df_beta_final.to_csv(out_beta, index=False)
print(f"Saved Stage-2 beta estimates to: {out_beta}")

# ---- Save MISE (all seeds + mean/se) ----
df_mise = pd.DataFrame({"seed": results["seeds"], "mise": results["mise_list"]})
mise_mean, _, mise_se = summarize_list(results["mise_list"])

df_mise_summary = pd.DataFrame(
    [
        {"seed": "mean", "mise": mise_mean},
        {"seed": "se", "mise": mise_se},
    ]
)

df_mise_final = pd.concat([df_mise, df_mise_summary], ignore_index=True)

out_mise = RESULTS_DIR / f"MISE_{ALGO_UP}_seed{first_seed}_to_seed{last_seed}.csv"
df_mise_final.to_csv(out_mise, index=False)
print(f"Saved Stage-2 MISE to: {out_mise}")



[Progress] Processing Simulation 1/100 (Seed: 1)


100%|██████████| 47/47 [00:44<00:00,  1.07it/s]


[Progress] Processing Simulation 2/100 (Seed: 2)



100%|██████████| 47/47 [00:36<00:00,  1.30it/s]


[Progress] Processing Simulation 3/100 (Seed: 3)


...


[Progress] Processing Simulation 99/100 (Seed: 99)



100%|██████████| 47/47 [00:35<00:00,  1.33it/s]


[Progress] Processing Simulation 100/100 (Seed: 100)



100%|██████████| 47/47 [00:37<00:00,  1.26it/s]



RESULTS (LASSO, K=100) - Second Stage Only
Mean MISE : 2.873244
Std MISE  : 2.391056
SE MISE   : 0.239106

Saved Stage-2 beta estimates to: /content/drive/MyDrive/Colab Notebooks/CTE_Codes/DML_methods/Data_and_Results/Estimates/estimates_LASSO_seed1_to_seed100.csv
Saved Stage-2 MISE to: /content/drive/MyDrive/Colab Notebooks/CTE_Codes/DML_methods/Data_and_Results/Estimates/MISE_LASSO_seed1_to_seed100.csv
